In [87]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle 
import pandas as pd
import numpy as np

In [88]:
# Load everything you saved during training
model = load_model('model.h5')

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('one_hot_encoder_geo.pkl', 'rb') as file:
    one_hot_encoder_geo = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [89]:
#input the data
input_data={
    'CreditScore':680,
    'Geography': 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure' :3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [90]:
input_df = pd.DataFrame([input_data])

In [ ]:
geo_encoded=one_hot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded,#the data 
    columns=one_hot_encoder_geo.get_feature_names_out(['Geography'])#just the column names
)
geo_encoded_df

/opt/anaconda3/envs/tfenv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [92]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,680,France,Male,40,3,60000,2,1,1,50000


In [93]:
#encode catogerical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,680,France,1,40,3,60000,2,1,1,50000


In [94]:
input_df = pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,680,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [95]:
#scaling the input
inputed_scaled=scaler.transform(input_df)
inputed_scaled

array([[ 0.29423332,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [96]:
prediction=model.predict(inputed_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step


array([[0.08568047]], dtype=float32)

In [97]:
prediction_prob=prediction[0][0]

In [98]:
prediction_prob

np.float32(0.08568047)

In [100]:
if prediction_prob>0.5:
    print('customer is likely to churn')
else:
    print('customer is not likely to churn')

customer is not likely to churn
